In [1]:
import datetime
from pathlib import Path
from typing import Final, Literal

import pandas as pd
import pycaret.regression as pyc

In [6]:
PROJECT_ROOT = Path.cwd().parent

DATASET = Literal["AGP", "GGMP"]

dataset:DATASET = "AGP"
# dataset:DATASET = "GGMP"

meta_path = PROJECT_ROOT / "datasets/processed" / dataset / "meta.tsv"
otu_path = PROJECT_ROOT / "datasets/processed" / dataset / "otu.tsv"

output_path = PROJECT_ROOT / "result" / dataset
output_path.mkdir(parents=True, exist_ok=True)

In [3]:
def split_otu_by_health(
    meta_path: Path, otu_path: Path
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    # Read meta.tsv and otu.tsv
    meta_df = pd.read_csv(meta_path, sep="\t")
    meta_df = meta_df.set_index("id")

    otu_df = pd.read_csv(otu_path, sep="\t")
    otu_df = otu_df.set_index("id")

    # Split otu_df based on the 'health' column in meta_df
    healthy_otu_df = otu_df[meta_df["health"] == "y"]
    # nonhealthy_otu_df = otu_df[meta_df["health"] == "n"]

    predicted_age_df = pd.merge(
        healthy_otu_df, meta_df["age"], left_index=True, right_index=True, how="inner"
    )

    return predicted_age_df, meta_df, otu_df


# Split otu.tsv into healthy and get predicted age dataframe
predicted_age_df, meta_df, otu_df = split_otu_by_health(meta_path, otu_path)


In [7]:
def model_health_ages(
    predicted_age_df: pd.DataFrame,
    otu_df: pd.DataFrame,
    output_dir: Path,
) -> pd.DataFrame:

    # Use pycaret to model healthy otu_df and predict the physiological age of the samples
    pyc.setup(
        data=predicted_age_df,
        target="age",
        session_id=123,
    )

    print("X Train Shape: ", pyc.get_config("X_train").shape)
    print("X Test Shape: ", pyc.get_config("X_test").shape)

    # Compare regression models and return the best model.
    best_model = pyc.compare_models(
        exclude=["lightgbm"],
        sort="MAE",
        errors="raise",
    )

    # pull() returns the comparison leaderboard as a DataFrame.
    compare_result = pyc.pull()
    compare_result.to_csv(
        output_dir / "compare_models.tsv",
        sep="\t",
        index=True,
    )

    # Tune the best model.
    tuned_model = pyc.tune_model(
        best_model,
        optimize="MAE",
        n_iter=50,
    )

    # pull() now returns the tuning results.
    tune_result = pyc.pull()
    tune_result.to_csv(
        output_dir / "tuned_best_model.tsv",
        sep="\t",
        index=True,
    )

    # Refit tuned model on the complete training dataset.
    final_best_model = pyc.finalize_model(tuned_model)

    # Predict physiological age.
    prediction_result = pyc.predict_model(
        final_best_model,
        data=otu_df,
    )
    age_predictions = prediction_result["prediction_label"]

    # Save final model.
    current_date = datetime.datetime.now(datetime.UTC).strftime("%Y%m%d")
    pyc.save_model(
        final_best_model,
        str(output_dir / f"final_best_model_{current_date}"),
    )

    return age_predictions.to_frame()


# Model healthy otu dataframe and predict ages
age_predictions = model_health_ages(predicted_age_df, otu_df, output_path)

,Description,Value
0,Session id,123
1,Target,age
2,Target type,Regression
3,Original data shape,"(1852, 1330)"
4,Transformed data shape,"(1852, 1330)"
5,Transformed train set shape,"(1296, 1330)"
6,Transformed test set shape,"(556, 1330)"
7,Numeric features,1329
8,Preprocess,True
9,Imputation type,simple


(1296, 1329)
(556, 1329)


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
catboost,CatBoost Regressor,10.1140,158.6354,12.5743,0.2667,0.2903,0.2582,6.6260
et,Extra Trees Regressor,10.3766,167.4752,12.9254,0.2268,0.2999,0.2678,1.1900
gbr,Gradient Boosting Regressor,10.5575,165.3362,12.8372,0.2354,0.2974,0.2702,0.9380
rf,Random Forest Regressor,10.6134,165.2226,12.8416,0.2365,0.2978,0.2732,1.7120
ada,AdaBoost Regressor,11.4606,185.4090,13.6084,0.1429,0.3176,0.2990,0.4380
dummy,Dummy Regressor,12.6323,220.7125,14.8460,-0.0196,0.3458,0.3313,0.0550
knn,K Neighbors Regressor,12.6481,234.6143,15.2732,-0.0821,0.3511,0.3246,0.0590
br,Bayesian Ridge,12.6795,227.3904,15.0526,-0.0486,0.3511,0.3314,0.1710
dt,Decision Tree Regressor,13.8709,324.5075,17.9753,-0.5004,0.4144,0.3404,0.1020
omp,Orthogonal Matching Pursuit,15.4368,1228.0434,29.0378,-4.5279,0.4240,0.3996,0.0660


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,10.8659,171.1042,13.0807,0.2470,0.3071,0.2812
1,10.9516,189.8134,13.7773,0.1052,0.2979,0.2532
2,10.3360,169.3258,13.0125,0.2189,0.3128,0.2872
3,9.8539,155.8451,12.4838,0.3110,0.2808,0.2485
4,9.0014,130.8666,11.4397,0.2523,0.2691,0.2401
5,10.3790,163.1424,12.7727,0.2473,0.3046,0.2804
6,11.1879,187.9928,13.7110,0.1297,0.2969,0.2597
7,10.7792,183.3574,13.5410,0.2783,0.3102,0.2727
8,9.7530,144.8769,12.0365,0.3158,0.2866,0.2602


Fitting 10 folds for each of 50 candidates, totalling 500 fits
Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).


Transformation Pipeline and Model Successfully Saved


In [8]:
def calculate_raw_gai(
    meta_df: pd.DataFrame, age_predictions: pd.DataFrame
) -> pd.DataFrame:
    """Add predicted age minus chronological age as the raw GAI."""
    meta_df["raw GAI"] = age_predictions["prediction_label"] - meta_df["age"]
    return meta_df

# Calculate raw GAI for all samples and add it to meta_df
meta_df = calculate_raw_gai(meta_df, age_predictions)

In [9]:
AGE_RANGES: Final[tuple[tuple[int, int], ...]] = (
    (18, 20),
    (20, 25),
    (25, 30),
    (30, 35),
    (35, 40),
    (40, 45),
    (45, 50),
    (50, 55),
    (55, 60),
    (60, 65),
    (65, 70),
    (70, 75),
    (75, 100),
)


def calculate_adjust_value(meta_df: pd.DataFrame, output_dir: Path) -> pd.DataFrame:
    """Calculate and assign the healthy-cohort adjustment for each age range."""
    healthy_raw_gai = meta_df.loc[meta_df["health"] == "y", "raw GAI"]

    adjust_values: list[float] = []
    for start_age, end_age in AGE_RANGES:
        in_age_range = (meta_df["age"] >= start_age) & (meta_df["age"] < end_age)
        adjust_values.append(healthy_raw_gai[in_age_range].mean())

    pd.DataFrame({"age_range": AGE_RANGES, "adjust_value": adjust_values}).to_csv(
        output_dir / "adjust_values.tsv", sep="\t", index=False
    )

    for (start_age, end_age), adjust_value in zip(
        AGE_RANGES,
        adjust_values,
        strict=True,
    ):
        in_age_range = (meta_df["age"] >= start_age) & (meta_df["age"] < end_age)
        meta_df.loc[in_age_range, "adjust value"] = adjust_value

    return meta_df



# Calculate adjust values based on age ranges and add them to meta_df
meta_df = calculate_adjust_value(meta_df, output_path)

In [10]:
def calculate_corrected_gai(meta_df):
    # Calculate corrected GAI by subtracting adjust value from raw GAI
    meta_df["corrected GAI"] = meta_df["raw GAI"] - meta_df["adjust value"]

    return meta_df


# Calculate corrected GAI and add it to meta_df
meta_df = calculate_corrected_gai(meta_df)

In [11]:

def save_result(meta_df: pd.DataFrame, result_path: Path) -> None:
    """Save the completed results table as a TSV file."""
    meta_df.to_csv(result_path, sep="\t", index=True)
    print(f"Saved result as {result_path}")

# Save final result as result.tsv
save_result(meta_df, output_path / "result.tsv")


Saved result as /home/kxviel/curahack-2026-challenge-5-main/result/AGP/result.tsv
